# Vallidate the Results of the Multi Unit-Regression 

The multi-unit regression uses building, parcel, and zillow data to calculate the units for multi-family homes when the provided unit is missing. To validate this data first we validate the output corresponds with the expected values. Then these values are validated at the census tract level to ensure the hosting capacity analysis is being completed on reliable data. 

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt

In [17]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

# import multifamily homes with units 
multi_zillow_w_units = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/final_zillow_w_volume/multi_zillow_w_units.parquet").to_crs(epsg=4326)

# import multifamily homes with units 
keep_outliers_multi_zillow_w_units = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/final_zillow_w_volume/keep_outliers_multi_zillow_w_units.parquet").to_crs(epsg=4326)

# import single family homes with  adjusted units 
single_condo_zillow_w_units = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/final_zillow_w_volume/single_condo_zillow_w_units.parquet").to_crs(epsg=4326)

### Validate the outputs of the unit data with the original Zillow dataset

In [3]:
print(f"The orginal Zillow data includes ", zillow.shape[0], "points")

The orginal Zillow data includes  10012620 points


In [14]:
# investigate the unit totals
multi_zillow = zillow[(zillow['type'] == 'Multi') & (zillow['code'] != 'RR106')]
single_zillow = zillow[zillow['type'] == 'Single']
condo_multi = zillow[(zillow['type'] == 'Multi') & (zillow['code'] == 'RR106')]

In [5]:
# condo data is crazy!
zillow[(zillow['type'] == 'Multi') & (zillow['code'] == 'RR106')]

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry
0,Multi,2003.0,1.0,None,None,I,224.0,491943.0,living,1003.0,3,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429)
1,Multi,2003.0,1.0,None,None,None,224.0,240117.0,living,936.0,4,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429)
2,Multi,2003.0,1.0,None,None,I,224.0,261770.0,living,997.0,5,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429)
3,Multi,2003.0,1.0,None,None,None,224.0,223337.0,living,1002.0,6,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429)
4,Multi,2003.0,1.0,None,None,None,224.0,241347.0,living,1003.0,7,06001403302,468,PGE/SCE,RR106,POINT (-122.26798 37.79407)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10005858,Multi,NaN,2.0,fossil,central,None,NaN,157620.0,living,810.0,10465825,06099000803,h2094,Others,RR106,POINT (-121.03261 37.67761)
10005859,Multi,NaN,2.0,fossil,central,None,NaN,63892.0,living,840.0,10465826,06099000803,h2094,Others,RR106,POINT (-121.03261 37.67709)
10005860,Multi,NaN,2.0,fossil,central,None,NaN,63892.0,living,840.0,10465827,06099000803,h2094,Others,RR106,POINT (-121.03261 37.67709)
10005861,Multi,NaN,2.0,fossil,central,None,NaN,123032.0,living,924.0,10465828,06099000803,h2094,Others,RR106,POINT (-121.03305 37.67760)


In [15]:
print(f"There are ", multi_zillow['unit'].sum() + single_zillow.shape[0] + condo_multi.shape[0], "units before the regression of multifamily homes.")

There are  13107748.0 units before the regression of multifamily homes.


In [7]:
print(f"The regressed Zillow data is", multi_zillow_w_units.shape[0] + single_condo_zillow_w_units.shape[0] ,"points")

The regressed Zillow data is 9706367 points


In [ ]:
print(f"The number of homes calculated in California by unit (when dropping outliers):", single_condo_zillow_w_units['total_unit'].sum() + multi_zillow_w_units['total_unit'].sum())

The number of homes calculated in the California by unit: 11041114.0


In [18]:
print(f'The number of homes calculated in California by unit (when keeping outliers): ', single_condo_zillow_w_units['total_unit'].sum() + keep_outliers_multi_zillow_w_units['total_unit'].sum())

The number of homes calculated in California by unit (when keeping outliers):  12505709.0


# Investigate differences

In [9]:
print(f"Maximum units for multi-family homes before the regression",multi_zillow['unit'].max())

Maximum units for multi-family homes before the regression 6857.0


In [10]:
print(f"Maximum units for the multi-family homes after the regression", multi_zillow_w_units['total_unit'].max())

Maximum units for the multi-family homes after the regression 3487.0


This difference in maximum units is huge. This could be part of the reason that we are getting fewer 

In [16]:
multi_zillow_w_units['total_unit'].sum()

1631920.0